# Day 3: Kafka Practice

## Welcome!
Use these solutions to check your work from `exercises/03_exercises.ipynb`. These solutions are for intermediate Kafka exercises, including Kafka operations and Wikimedia stream processing.

## Before You Start
- Run `docker-compose up zookeeper kafka` in the project folder using the updated `docker-compose.yml` (with Kafka 7.5.3 and Zookeeper 7.5.3).
- Open Jupyter Notebook at http://localhost:8888.
- Install `kafka-python` if needed: `!pip install kafka-python`.
- Install `requests` for the Wikimedia stream: `!pip install requests`.

---

### Exercise 1: Create a Topic
#### What to Do
- Create a Kafka topic named `test-topic` with 1 partition and replication factor 1 using `kafka-python`.

In [3]:
!pip install kafka-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 326.3/326.3 kB 4.1 MB/s eta 0:00:00a 0:00:01


In [2]:
from kafka.admin import KafkaAdminClient, NewTopic
from kafka.errors import KafkaError

# Create admin client
admin_client = KafkaAdminClient(bootstrap_servers="host.docker.internal:9093")

# Create topic
try:
    topic_list = [NewTopic(name="test-topic3", num_partitions=1, replication_factor=1)]
    admin_client.create_topics(new_topics=topic_list)
except KafkaError as e:
    print(f"Error creating topic: {e}")

# Verify
print(admin_client.list_topics())

['__consumer_offsets', '_connect-offsets', '_connect-status', '_confluent-ksql-ksql-service_command_topic', '_connect-configs', 'test-topic', 'test-topic3']


---

### Exercise 2: List Topics
#### What to Do
- List all Kafka topics using `kafka-python`.

In [3]:
# List topics
print(admin_client.list_topics())

['__consumer_offsets', '_connect-offsets', '_connect-status', '_confluent-ksql-ksql-service_command_topic', '_connect-configs', 'test-topic', 'test-topic3']


---

### Exercise 3: Produce a Message
#### What to Do
- Send a message "Hello Kafka!" to `test-topic` using `kafka-python`.

In [62]:
from kafka import KafkaProducer

# Create producer
producer = KafkaProducer(bootstrap_servers="host.docker.internal:9093")

# Send message
producer.send('test-topic', b'Hello Kafka!')
producer.flush()

print("Message sent!")

Message sent!


---

### Exercise 4: Produce Multiple Messages
#### What to Do
- Send three messages to `test-topic` using `kafka-python`.

In [36]:
# Send multiple messages
messages = [b'Message 1', b'Message 2', b'Message L']
for message in messages:
    producer.send('test-topic', message)
    print(f"Sent: {message.decode('utf-8')}")
producer.flush()

print("Three messages sent!")

Sent: Message 1
Sent: Message 2
Sent: Message L
Three messages sent!


---

### Exercise 5: Consume a Message
#### What to Do
- Read one message from `test-topic` using `kafka-python`.

In [37]:
from kafka import KafkaConsumer

# Create consumer
consumer = KafkaConsumer('test-topic', bootstrap_servers="host.docker.internal:9093", auto_offset_reset='earliest')

# Consume one message
for message in consumer:
    print(message.value.decode('utf-8'))
    break

Hello Kafka!


---

### Exercise 6: Consume Multiple Messages
#### What to Do
- Read three messages from `test-topic` using `kafka-python`.

In [38]:
# Consume three messages
message_count = 0
for message in consumer:
    print(message.value.decode('utf-8'))
    message_count += 1
    if message_count == 3:
        break

Message 1
Message 2
Message L


---

### Exercise 7: Create a Partitioned Topic
#### What to Do
- Create a topic named `partitioned-topic` with 3 partitions and replication factor 1 using `kafka-python`.

In [9]:
# Create topic with 3 partitions
try:
    topic_list = [NewTopic(name="partitioned-topic", num_partitions=3, replication_factor=1)]
    admin_client.create_topics(new_topics=topic_list)
except KafkaError as e:
    print(f"Error creating topic: {e}")

# Verify
print(admin_client.list_topics())

['__consumer_offsets', 'partitioned-topic', '_connect-offsets', '_connect-status', '_confluent-ksql-ksql-service_command_topic', '_connect-configs', 'test-topic', 'test-topic3']


---

### Exercise 8: Produce to Specific Partition
#### What to Do
- Send a message to partition 1 of `partitioned-topic` using `kafka-python`.

In [63]:
# Send to partition 1
producer.send('partitioned-topic', b'Message sent for Partition 1', partition=1)
producer.flush()

print("Message sent to partition 1")

producer.send('partitioned-topic', b'Message sent for Partition 0', partition=0)
producer.flush()

print("Message sent to partition 0")


Message sent to partition 1
Message sent to partition 0


---

### Exercise 9: Consume from Specific Partition
#### What to Do
- Read one message from partition 1 of `partitioned-topic` using `kafka-python`.

In [11]:
from kafka import TopicPartition

# Consume from partition 1
consumer = KafkaConsumer(bootstrap_servers="host.docker.internal:9093", auto_offset_reset='earliest')
consumer.assign([TopicPartition('partitioned-topic', 1)])

for message in consumer:
    print(message.value.decode('utf-8'))
    break

Message to Partition 1


---

### Exercise 10: Delete a Topic
#### What to Do
- Delete the `test-topic` using `kafka-python`.

In [70]:
# Delete topic
try:
    admin_client.delete_topics(topics=["wiki-aggregates"])
except KafkaError as e:
    print(f"Error deleting topic: {e}")

# Verify
print(admin_client.list_topics())

['test-topic', 'retention-topic', '__consumer_offsets', 'partitioned-topic', '_connect-offsets', '_connect-status', '_confluent-ksql-ksql-service_command_topic', '_connect-configs']


---

### Exercise 11: Connect to Wikimedia Stream
#### What to Do
- Connect to the Wikipedia recent changes stream and print the first message using a custom SSE client.

In [15]:
import requests
import json
from datetime import datetime

def print_event(event, show_bots=False, languages=None):
    """Print basic information about a Wikipedia event."""
    if event.get('bot') and not show_bots:
        return False
    if languages and event.get('wiki') not in languages:
        return False
    title = event.get('title', 'N/A')
    user = event.get('user', 'Anonymous')
    comment = event.get('comment', '')
    wiki = event.get('wiki', 'N/A')
    event_type = event.get('type', 'edit')
    ts = event.get('meta', {}).get('dt', '')
    time_str = datetime.fromisoformat(ts.replace('Z', '+00:00')).strftime('%H:%M:%S') if ts else ''
    print(f"Wiki: {wiki}")
    print(f"Title: {title}")
    print(f"User: {user}")
    print(f"Type: {event_type}")
    print(f"Time: {time_str}")
    if comment:
        print(f"Comment: {comment[:80]}{'...' if len(comment)>80 else ''}")
    print("-" * 50)
    return True

def main():
    url = 'https://stream.wikimedia.org/v2/stream/recentchange'
    headers = {'User-Agent': 'WikiStreamBot/1.0', 'Accept': 'text/event-stream'}
    show_bots = False
    languages = ['enwiki']
    print("Wikipedia Live Changes (Press Ctrl+C to stop)\n")
    try:
        with requests.get(url, headers=headers, stream=True) as resp:
            for line in resp.iter_lines(decode_unicode=True):
                if line.startswith('data: '):
                    try:
                        event = json.loads(line[6:])
                        if event.get('meta', {}).get('domain') == 'canary':
                            continue
                        if print_event(event, show_bots, languages):
                            break
                    except json.JSONDecodeError:
                        continue
    except KeyboardInterrupt:
        print("Stopped by user")

if __name__ == "__main__":
    main()

Wikipedia Live Changes (Press Ctrl+C to stop)

Wiki: enwiki
Title: Category:Chilean bankers
User: Carigval.97
Type: categorize
Time: 21:53:37
Comment: [[:Enrique Aguirre Pinto]] added to category
--------------------------------------------------


---

### Exercise 12: Extract User from Wikimedia Stream
#### What to Do
- Connect to the Wikipedia recent changes stream and print the `user` field from the first 3 events.

In [22]:
import requests
import json

def print_user(event, user_count=[0]):
    """Print the user field and track count."""
    if user_count[0] < 3:
        user = event.get('user', 'Anonymous')
        print(f"User: {user}")
        user_count[0] += 1
        return user_count[0] < 3
    return False

def main():
    url = 'https://stream.wikimedia.org/v2/stream/recentchange'
    headers = {'User-Agent': 'WikiStreamBot/1.0', 'Accept': 'text/event-stream'}
    print("Extracting Users from Wikipedia Changes (Press Ctrl+C to stop)\n")
    try:
        with requests.get(url, headers=headers, stream=True) as resp:
            for line in resp.iter_lines(decode_unicode=True):
                if line.startswith('data: '):
                    try:
                        event = json.loads(line[6:])
                        if event.get('meta', {}).get('domain') == 'canary':
                            continue
                        if not print_user(event):
                            break
                    except json.JSONDecodeError:
                        continue
    except KeyboardInterrupt:
        print("Stopped by user")

if __name__ == "__main__":
    main()

Extracting Users from Wikipedia Changes (Press Ctrl+C to stop)

User: PantheraLeo1359531
User: Mdewman6
User: Rathfelder


---

### Exercise 13: Count Edits in Wikimedia Stream
#### What to Do
- Connect to the Wikipedia recent changes stream and count the first 10 events of type `edit`.

In [28]:
import requests
import json

def count_edits(event, counters=[0, 0]):
    """Count edits and total events."""
    counters[0] += 1
    if event.get('type') == 'edit':
        counters[1] += 1
    print(f"Total events: {counters[0]}, Edit events: {counters[1]}")
    return counters[0] < 10

def main():
    url = 'https://stream.wikimedia.org/v2/stream/recentchange'
    headers = {'User-Agent': 'WikiStreamBot/1.0', 'Accept': 'text/event-stream'}
    counters = [0, 0]
    print("Counting Edits from Wikipedia Changes (Press Ctrl+C to stop)\n")
    try:
        with requests.get(url, headers=headers, stream=True) as resp:
            for line in resp.iter_lines(decode_unicode=True):
                if line.startswith('data: '):
                    try:
                        event = json.loads(line[6:])
                        if event.get('meta', {}).get('domain') == 'canary':
                            continue
                        if not count_edits(event, counters):
                            print(f"Final count - Total events: {counters[0]}, Edit events: {counters[1]}")
                            break
                    except json.JSONDecodeError:
                        continue
    except KeyboardInterrupt:
        print("Stopped by user")

if __name__ == "__main__":
    main()

Counting Edits from Wikipedia Changes (Press Ctrl+C to stop)

Total events: 1, Edit events: 1
Total events: 2, Edit events: 1
Total events: 3, Edit events: 2
Total events: 4, Edit events: 2
Total events: 5, Edit events: 2
Total events: 6, Edit events: 3
Total events: 7, Edit events: 4
Total events: 8, Edit events: 5
Total events: 9, Edit events: 5
Total events: 10, Edit events: 6
Final count - Total events: 6, Edit events: 10


---

### Exercise 14: Configure Topic with Custom Retention and Segment Size
#### What to Do
- Create a topic named `retention-topic` with a retention period of 2 hours and a segment size of 10MB.

In [29]:
from kafka import KafkaAdminClient
from kafka.admin import NewTopic
from kafka.errors import TopicAlreadyExistsError

# Connect to Kafka broker running in Docker container
print("Connecting to Kafka broker")
admin_client = KafkaAdminClient(
    bootstrap_servers="host.docker.internal:9093",  # Docker container access
    client_id='retention_topic_creator'
)

# Configure topic retention and segment settings
print("Setting up topic configuration")
topic_configs = {
    'retention.ms': str(2 * 60 * 60 * 1000),    # 2 hours = 7,200,000 ms
    'segment.bytes': str(10 * 1024 * 1024),     # 10MB = 10,485,760 bytes
    'cleanup.policy': 'delete'                   # Delete old segments when expired
}

# Create topic object with configurations
new_topic = NewTopic(
    name='retention-topic',           # Topic name
    num_partitions=1,                # Single partition for simplicity
    replication_factor=1,            # Single replica (no redundancy)
    topic_configs=topic_configs      # Apply retention and segment settings
)

# Attempt to create the topic
print("Creating topic 'retention-topic'")
try:
    admin_client.create_topics(new_topics=[new_topic], validate_only=False)
    print("Topic created successfully")
except TopicAlreadyExistsError:
    print("Topic already exists - skipping creation")
except Exception as e:
    print(f"Creation failed: {e}")

print("Done")


Connecting to Kafka broker
Setting up topic configuration
Creating topic 'retention-topic'
Topic created successfully
Done


---

### Exercise 15: Produce JSON Messages with Custom Serializer
#### What to Do
- Send 3 JSON messages to `retention-topic`, each containing fields `event_id`, `category`, and `timestamp`, using a custom JSON serializer.

In [30]:
import json
from datetime import datetime

# Custom JSON serializer
def json_serializer(data):
    return json.dumps(data).encode('utf-8')

# Create producer with serializer
producer = KafkaProducer(bootstrap_servers="host.docker.internal:9093", value_serializer=json_serializer)

# Send JSON messages
messages = [
    {"event_id": 1, "category": "sale", "timestamp": "2025-09-18T10:00:00"},
    {"event_id": 2, "category": "update", "timestamp": "2025-09-18T10:01:00"},
    {"event_id": 3, "category": "sale", "timestamp": "2025-09-18T10:02:00"}
]
for msg in messages:
    producer.send('retention-topic', msg)
    print(f"Sent: {msg}")
producer.flush()
print("3 JSON messages sent!")

Sent: {'event_id': 1, 'category': 'sale', 'timestamp': '2025-09-18T10:00:00'}
Sent: {'event_id': 2, 'category': 'update', 'timestamp': '2025-09-18T10:01:00'}
Sent: {'event_id': 3, 'category': 'sale', 'timestamp': '2025-09-18T10:02:00'}
3 JSON messages sent!


---

### Exercise 16: Consume and Validate JSON Messages
#### What to Do
- Read 3 JSON messages from `retention-topic`, validate that they contain required fields, and print the `category` field.

In [31]:
import json

# Consume and validate JSON
consumer = KafkaConsumer('retention-topic', bootstrap_servers="host.docker.internal:9093", auto_offset_reset='earliest')
required_fields = ["event_id", "category", "timestamp"]
count = 0
for message in consumer:
    try:
        data = json.loads(message.value.decode('utf-8'))
        if all(field in data for field in required_fields):
            print(f"Category: {data['category']}")
            count += 1
        else:
            print("Invalid message: missing required fields")
        if count == 3:
            break
    except json.JSONDecodeError:
        print("JSON parse error")
        continue
consumer.close()

Category: sale
Category: update
Category: sale


---

### Exercise 17: Distribute Messages Across Consumer Group
#### What to Do
- Simulate two consumers in the same consumer group reading from `partitioned-topic` (3 partitions) and print the partition and message for each.

In [ ]:
# Consumer 1
consumer1 = KafkaConsumer(
    'partitioned-topic',
    bootstrap_servers="host.docker.internal:9093",
    group_id='test-group',
    auto_offset_reset='earliest'
)
print("Consumer 1 started...")
for message in consumer1:
    print(f"Consumer 1, Partition {message.partition}: {message.value.decode('utf-8')}")
    break

# Note: Run Consumer 2 in a separate cell or script
consumer2 = KafkaConsumer(
    'partitioned-topic',
    bootstrap_servers="host.docker.internal:9093",
    group_id='test-group',
    auto_offset_reset='earliest'
)
print("Consumer 2 started...")
for message in consumer2:
    print(f"Consumer 2, Partition {message.partition}: {message.value.decode('utf-8')}")
    break

---

### Exercise 18: Filter Wikimedia Stream by Edit Size
#### What to Do
- Connect to the Wikipedia stream and print the first 3 edit events where the edit size (`new_bytes - old_bytes`) is greater than 100 bytes.

In [45]:
import requests
import json

def print_large_edit(event, count=[0]):
    """Print events with edit size > 100 bytes."""
    if count[0] < 3 and event.get('type') == 'edit':
        new_bytes = event.get('length', {}).get('new', 0)
        old_bytes = event.get('length', {}).get('old', 0)
        edit_size = new_bytes - old_bytes
        if edit_size > 100:
            count[0] += 1
            print(f"Event {count[0]} - Title: {event.get('title', 'N/A')}, User: {event.get('user', 'Anonymous')}, Edit Size: {edit_size}")
            return count[0] < 3
    return True

def main():
    url = 'https://stream.wikimedia.org/v2/stream/recentchange'
    headers = {'User-Agent': 'WikiStreamBot/1.0', 'Accept': 'text/event-stream'}
    print("Filtering Large Edits (Press Ctrl+C to stop)\n")
    try:
        with requests.get(url, headers=headers, stream=True) as resp:
            for line in resp.iter_lines(decode_unicode=True):
                if line.startswith('data: '):
                    try:
                        event = json.loads(line[6:])
                        if event.get('meta', {}).get('domain') == 'canary':
                            continue
                        if not print_large_edit(event):
                            break
                    except json.JSONDecodeError:
                        continue
    except KeyboardInterrupt:
        print("Stopped by user")

if __name__ == "__main__":
    main()

Filtering Large Edits (Press Ctrl+C to stop)

Event 1 - Title: File:1252021 I Montacute House, Great Hall Montacute 20250830 0016.jpg, User: SchlurcherBot, Edit Size: 2338
Event 2 - Title: Q136712912, User: SabrinaMacGrg, Edit Size: 351
Event 3 - Title: aberration, User: Bot-Jagwar, Edit Size: 205


---

### Exercise 19: Produce Key-Based Messages with Consistent Partitioning
#### What to Do
- Send 5 messages to `partitioned-topic` with keys `user1` and `user2` to ensure messages with the same key go to the same partition.

In [60]:
import json
from kafka import KafkaProducer

producer = KafkaProducer(
    bootstrap_servers='host.docker.internal:9093',
    key_serializer=lambda x: json.dumps(x).encode('utf-8'),
    value_serializer=lambda x: json.dumps(x).encode('utf-8')
)

# Use keys that are more likely to distribute across partitions
messages = [
    ('alice', 'Message 1 from alice'),
    ('bob', 'Message 1 from bob'),
    ('alice', 'Message 2 from alice'),     # Same partition as first alice
    ('charlie', 'Message 1 from charlie'),
    ('bob', 'Message 2 from bob'),         # Same partition as first bob
    ('alice', 'Message 3 from alice')      # Same partition as other alice messages
]

for key, value in messages:
    future = producer.send('partitioned-topic', key=key, value=value)
    print(f"Sent message with key {key} to partition {future.get().partition}")

producer.flush()


Sent message with key alice to partition 0
Sent message with key bob to partition 2
Sent message with key alice to partition 0
Sent message with key charlie to partition 2
Sent message with key bob to partition 2
Sent message with key alice to partition 0


---

### Exercise 20: Consume with Manual Offset Commit
#### What to Do
- Create a consumer for `partitioned-topic` that manually commits offsets after processing each message.

In [67]:
from kafka import KafkaConsumer

consumer = KafkaConsumer(
    'partitioned-topic',
    bootstrap_servers='host.docker.internal:9093',
    group_id='my-consumer-group',           # Required for commit()
    auto_offset_reset='earliest',
    enable_auto_commit=False,               # Disable auto-commit for manual control
    value_deserializer=lambda x: x.decode('utf-8')
)

count = 0
for message in consumer:
    count += 1
    if count < 3:
        print(f"Received: {message.value}")
        consumer.commit()  # Now this works
        print(f"Committed offset for partition {message.partition}")
    else:
        break
consumer.close()


Received: "Message 2 from alice"
Committed offset for partition 0
Received: "Message 3 from alice"
Committed offset for partition 0


---

### Exercise 21: Stream and Aggregate Wikimedia Events to Kafka
#### What to Do
- Connect to the Wikipedia stream, aggregate edit counts by `wiki` for the first 10 events, and produce the aggregated data to a new topic `wiki-aggregates`.

In [71]:
from kafka.admin import NewTopic
import requests
import json

# Create topic
try:
    topic_list = [NewTopic(name="wiki-aggregates", num_partitions=1, replication_factor=1)]
    admin_client.create_topics(new_topics=topic_list)
except KafkaError as e:
    print(f"Error creating topic: {e}")

# Producer
producer = KafkaProducer(bootstrap_servers="host.docker.internal:9093", value_serializer=lambda x: json.dumps(x).encode('utf-8'))

# Aggregate and stream
url = 'https://stream.wikimedia.org/v2/stream/recentchange'
headers = {'User-Agent': 'WikiStreamBot/1.0', 'Accept': 'text/event-stream'}
edit_counts = {}
total_events = 0
print("Aggregating Wikimedia Edits (Press Ctrl+C to stop)\n")
try:
    with requests.get(url, headers=headers, stream=True) as resp:
        for line in resp.iter_lines(decode_unicode=True):
            if line.startswith('data: '):
                try:
                    event = json.loads(line[6:])
                    if event.get('meta', {}).get('domain') == 'canary':
                        continue
                    total_events += 1
                    if event.get('type') == 'edit':
                        wiki = event.get('wiki', 'unknown')
                        edit_counts[wiki] = edit_counts.get(wiki, 0) + 1
                    if total_events == 10:
                        producer.send('wiki-aggregates', {'edit_counts': edit_counts, 'total_events': total_events})
                        print(f"Sent aggregates: {edit_counts}")
                        break
                except json.JSONDecodeError:
                    continue
    producer.flush()
    print("Aggregation complete.")
except KeyboardInterrupt:
    print("Stopped by user")

Aggregating Wikimedia Edits (Press Ctrl+C to stop)

Sent aggregates: {'enwiktionary': 1, 'commonswiki': 5, 'wikidatawiki': 1}
Aggregation complete.


---
### Exercise 22: Stream and Aggregate Wikimedia Events to Kafka Topic
#### What to Do
- Consume from the wiki-aggregates topic to verify the aggregated data.

In [72]:
from kafka import KafkaConsumer, TopicPartition
import json

print("Creating consumer without consumer group...")

consumer = KafkaConsumer(
    bootstrap_servers="host.docker.internal:9093",
    consumer_timeout_ms=10000,  # 10 second timeout
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

# Manual partition assignment - always reads from beginning
partitions = [TopicPartition('wiki-aggregates', 0)]
consumer.assign(partitions)
consumer.seek_to_beginning()  # Always start from beginning

print("Reading from wiki-aggregates topic:\n")

try:
    message_count = 0
    for message in consumer:
        message_count += 1
        data = message.value
        print(f"Message {message_count}:")
        print(f"Edit counts: {data['edit_counts']}")
        print(f"Total events: {data['total_events']}")
        print(f"Partition: {message.partition}, Offset: {message.offset}")
        print("-" * 50)
    
    print(f"Total messages read: {message_count}")
    
except KeyboardInterrupt:
    print("\nStopped by user")
except Exception as e:
    print(f"Error: {e}")
finally:
    consumer.close()
    print("Consumer closed")


Creating consumer without consumer group...
Reading from wiki-aggregates topic:

Message 1:
Edit counts: {'enwiktionary': 1, 'commonswiki': 5, 'wikidatawiki': 1}
Total events: 10
Partition: 0, Offset: 0
--------------------------------------------------
Total messages read: 1
Consumer closed


## Notes
- All exercises assume Kafka and Zookeeper are running with the provided `docker-compose.yml` (Kafka 7.5.3 and Zookeeper 7.5.3).
- Use `print()` to verify outputs in Jupyter Notebook.
- If errors occur, ensure Kafka is running with `docker ps` and verify the connection string (`host.docker.internal:9093`).